In [22]:
# -----------------------------------------------------------
# CELL 1: SETUP & VISUALIZATION TOOLS
# -----------------------------------------------------------
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import pickle
import json
from datetime import datetime
from google.colab import drive

# 1. Mount Drive
if os.path.ismount('/content/drive'):
    print("✓ Drive already mounted.")
else:
    drive.mount('/content/drive', force_remount=True)

# 2. Paths
BASE_DIR = "/content/drive/MyDrive/FabricDefect"
FEATURE_DIR = f"{BASE_DIR}/features"
MODEL_DIR = f"{BASE_DIR}/models"
BACKUP_DIR = f"{BASE_DIR}/backups"

# 3. Plotting Config
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

# --- CUSTOM METRICS (NO SKLEARN) ---
def get_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def get_confusion_matrix(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t][p] += 1
    return cm

def print_classification_report(y_true, y_pred, class_names):
    num_classes = len(class_names)
    cm = get_confusion_matrix(y_true, y_pred, num_classes)
    print(f"{'Class':<25} {'Precision':<10} {'Recall':<10} {'F1-Score':<10}")
    print("-" * 60)

    precisions = []
    recalls = []
    f1s = []
    supports = []

    for i in range(num_classes):
        tp = cm[i, i]
        fp = np.sum(cm[:, i]) - tp
        fn = np.sum(cm[i, :]) - tp
        support = np.sum(cm[i, :])
        supports.append(support)

        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2*p*r / (p+r) if (p+r) > 0 else 0

        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)

        print(f"{class_names[i]:<25} {p:.4f}     {r:.4f}     {f1:.4f}")

    print("-" * 60)

    # Weighted Avg
    total = np.sum(supports)
    w_p = np.sum(np.array(precisions) * np.array(supports)) / total
    w_r = np.sum(np.array(recalls) * np.array(supports)) / total
    w_f1 = np.sum(np.array(f1s) * np.array(supports)) / total

    print(f"{'Weighted Avg':<25} {w_p:.4f}     {w_r:.4f}     {w_f1:.4f}")

def plot_confusion_matrix(cm, classes, title):
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    ax.set(xticks=np.arange(cm.shape[1]), yticks=np.arange(cm.shape[0]),
           xticklabels=classes, yticklabels=classes,
           title=title, ylabel='True Label', xlabel='Predicted Label')
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], 'd'),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    plt.show()

print("✓ Custom Tools Loaded.")

✓ Drive already mounted.
✓ Custom Tools Loaded.


In [23]:
# -----------------------------------------------------------
# CELL 2: LOAD DATA
# -----------------------------------------------------------
print("Loading data...")
X_test  = np.load(f"{FEATURE_DIR}/final_X_test.npy")
y_test  = np.load(f"{FEATURE_DIR}/final_y_test.npy")
X_val   = np.load(f"{FEATURE_DIR}/final_X_val.npy")
y_val   = np.load(f"{FEATURE_DIR}/final_y_val.npy")

# Load Metadata
with open(f"{FEATURE_DIR}/dataset_metadata.pkl", "rb") as f:
    meta = pickle.load(f)
class_names = list(meta['class_to_idx'].keys())
NUM_CLASSES = len(class_names)

print(f"Test Data: {X_test.shape}")
print(f"Val Data:  {X_val.shape}")
print(f"Classes: {class_names}")

# --- Helper for One-Hot ---
def to_one_hot(y, n_classes):
    n_samples = y.shape[0]
    one_hot = np.zeros((n_samples, n_classes))
    one_hot[np.arange(n_samples), y] = 1
    return one_hot

Loading data...
Test Data: (548, 2048)
Val Data:  (435, 2048)
Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [24]:
# -----------------------------------------------------------
# CELL 3: REDEFINE CUSTOM CLASSES
# -----------------------------------------------------------
import numpy as np

# --- Helper ---
def softmax(z):
    z -= np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

# --- TREE NODE ---
class TreeNode:
    def __init__(self, depth=0):
        self.depth = depth
        self.left = None
        self.right = None
        self.feature_idx = None
        self.threshold = None
        self.value = None
        self.is_leaf = False

class Node:
    def __init__(self, value=None):
        self.value = value

# --- REGULARIZED TREE ---
class RegularizedTree:
    def __init__(self, max_depth=3, lambda_reg=1.0, gamma=0.0, min_child_weight=1.0, feature_subsample=1.0):
        self.max_depth = max_depth
        self.lambda_reg = lambda_reg
        self.gamma = gamma
        self.root = None

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])

    def _traverse(self, x, node):
        if node.is_leaf: return node.value
        if x[node.feature_idx] <= node.threshold: return self._traverse(x, node.left)
        else: return self._traverse(x, node.right)

# --- XGBOOST ---
class CustomXGBoost:
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3, reg_lambda=1.0, gamma=0.0):
        self.K = n_estimators
        self.lr = learning_rate
        self.trees = []

    def predict_proba(self, X):
        if not hasattr(self, 'n_classes'): self.n_classes = len(self.trees[0])
        scores = np.zeros((X.shape[0], self.n_classes))
        for iter_trees in self.trees:
            for k, tree in enumerate(iter_trees):
                scores[:, k] += self.lr * tree.predict(X)
        return softmax(scores)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

# --- LIGHTGBM ---
class CustomLightGBM:
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3, goss_alpha=0.2, goss_beta=0.2):
        self.K = n_estimators
        self.lr = learning_rate
        self.trees = []

    def predict_proba(self, X):
        if not hasattr(self, 'n_classes'): self.n_classes = len(self.trees[0])
        scores = np.zeros((X.shape[0], self.n_classes))
        for iter_trees in self.trees:
            for k, tree in enumerate(iter_trees):
                scores[:, k] += self.lr * tree.predict(X)
        return softmax(scores)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

# --- CATBOOST ---
class CustomCatBoost:
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3):
        self.K = n_estimators
        self.lr = learning_rate
        self.trees = []

    def predict_proba(self, X):
        if not hasattr(self, 'n_classes'): self.n_classes = len(self.trees[0])
        scores = np.zeros((X.shape[0], self.n_classes))
        for iter_trees in self.trees:
            for k, tree in enumerate(iter_trees):
                scores[:, k] += self.lr * tree.predict(X)
        return softmax(scores)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

print("✓ Model Classes Defined.")

✓ Model Classes Defined.


In [25]:
# -----------------------------------------------------------
# CELL 4: LOAD PRE-TRAINED MODELS
# -----------------------------------------------------------
import glob

# UPDATE THIS FOLDER NAME TO YOUR BACKUP
BACKUP_FOLDER_NAME = "custom_run_20251205_001120"
load_path = os.path.join(BACKUP_DIR, BACKUP_FOLDER_NAME)

print(f"📂 Loading from: {load_path}")

if not os.path.exists(load_path):
    print("❌ Error: Directory not found.")
else:
    all_files = glob.glob(os.path.join(load_path, "*.pkl"))
    models = {}

    for filepath in all_files:
        filename = os.path.basename(filepath).lower()
        try:
            with open(filepath, "rb") as f:
                loaded_obj = pickle.load(f)

            if "xgb" in filename: models["CustomXGB"] = loaded_obj
            elif "lgb" in filename: models["CustomLGBM"] = loaded_obj
            elif "cat" in filename: models["CustomCatB"] = loaded_obj
            print(f"   ✅ Loaded {filename}")
        except Exception as e:
            print(f"   ❌ Failed {filename}: {e}")

    xgb_model = models["CustomXGB"]
    lgb_model = models["CustomLGBM"]
    cat_model = models["CustomCatB"]
    print("\n🚀 All Base Models Loaded!")

📂 Loading from: /content/drive/MyDrive/FabricDefect/backups/custom_run_20251205_001120
   ✅ Loaded custom_xgb.pkl
   ✅ Loaded custom_lgbm.pkl
   ✅ Loaded custom_catb.pkl

🚀 All Base Models Loaded!


In [26]:
# -----------------------------------------------------------
# CELL 5: GENERATE META-FEATURES
# -----------------------------------------------------------
print("Generating Meta-Features (Probabilities)...")

# 1. Validation Set (Used to train the Stacker)
#    The Stacker learns: "When XGB says Class 1 with 0.8 prob, it's usually right"
meta_X_val = np.hstack([
    xgb_model.predict_proba(X_val),
    lgb_model.predict_proba(X_val),
    cat_model.predict_proba(X_val)
])

# 2. Test Set (Used to evaluate the Stacker)
meta_X_test = np.hstack([
    xgb_model.predict_proba(X_test),
    lgb_model.predict_proba(X_test),
    cat_model.predict_proba(X_test)
])

print(f"Meta-Features Shape (Val): {meta_X_val.shape}")
print(f"Meta-Features Shape (Test): {meta_X_test.shape}")
print("✓ Ready for Stacking.")

Generating Meta-Features (Probabilities)...
Meta-Features Shape (Val): (435, 18)
Meta-Features Shape (Test): (548, 18)
✓ Ready for Stacking.


In [19]:
# -----------------------------------------------------------
# CELL 6: EVALUATE INDIVIDUAL MODELS
# -----------------------------------------------------------
print("\n--- INDIVIDUAL MODEL PERFORMANCE (TEST SET) ---")
individual_acc = {}

for name, probs in test_preds.items():
    preds = np.argmax(probs, axis=1)
    acc = get_accuracy(y_test, preds)
    individual_acc[name] = acc
    print(f"{name:<10} Accuracy: {acc:.2%}")


--- INDIVIDUAL MODEL PERFORMANCE (TEST SET) ---
XGB        Accuracy: 92.34%
LGB        Accuracy: 92.34%
Cat        Accuracy: 92.52%


In [34]:
# -----------------------------------------------------------
# CELL 6: WEIGHT OPTIMIZATION STACKER (FROM SCRATCH)
# -----------------------------------------------------------

class WeightOptimizationStacker:
    """
    Finds the optimal weights [w1, w2, w3] to combine predictions.
    Uses Random Search to maximize validation accuracy.
    """
    def __init__(self, n_trials=5000):
        self.n_trials = n_trials
        self.best_weights = None
        self.best_acc = 0.0

    def fit(self, preds_list, y_true):
        """
        preds_list: List of probability arrays [P_xgb, P_lgb, P_cat]
        y_true: Actual labels
        """
        print(f"Optimizing Ensemble Weights ({self.n_trials} trials)...")

        n_models = len(preds_list)
        best_w = np.ones(n_models) / n_models # Start equal

        for i in range(self.n_trials):
            # 1. Generate Random Weights
            # Create random numbers and normalize them to sum to 1
            weights = np.random.rand(n_models)
            weights /= np.sum(weights)

            # 2. Weighted Average of Probabilities
            # P_final = w1*P1 + w2*P2 + w3*P3
            weighted_sum = np.zeros_like(preds_list[0])
            for p, w in zip(preds_list, weights):
                weighted_sum += p * w

            # 3. Calculate Accuracy
            final_preds = np.argmax(weighted_sum, axis=1)
            acc = np.mean(final_preds == y_true)

            # 4. Update Best
            if acc > self.best_acc:
                self.best_acc = acc
                self.best_weights = weights

        print(f"✓ Optimization Complete.")
        print(f"  Best Val Accuracy: {self.best_acc:.4%}")
        print(f"  Optimal Weights: {self.best_weights}")

    def predict(self, preds_list):
        # Apply learned weights
        weighted_sum = np.zeros_like(preds_list[0])
        for p, w in zip(preds_list, self.best_weights):
            weighted_sum += p * w
        return np.argmax(weighted_sum, axis=1)

print("✓ Weight Optimizer Defined.")

✓ Weight Optimizer Defined.


In [35]:
# -----------------------------------------------------------
# CELL 7: RUN OPTIMIZATION
# -----------------------------------------------------------

# 1. Prepare Validation Predictions list
# We use the validation set to find the best weights
val_preds_list = [
    xgb_model.predict_proba(X_val),
    lgb_model.predict_proba(X_val),
    cat_model.predict_proba(X_val)
]

# 2. Run Optimization
stacker = WeightOptimizationStacker(n_trials=10000) # 10k iterations is fast in NumPy
stacker.fit(val_preds_list, y_val)

# 3. Visualize Weights
plt.figure(figsize=(6, 4))
plt.bar(["XGBoost", "LightGBM", "CatBoost"], stacker.best_weights, color=['#4C72B0', '#55A868', '#C44E52'])
plt.title("Optimal Ensemble Weights")
plt.ylabel("Weight Contribution")
plt.ylim(0, 1.0)
for i, v in enumerate(stacker.best_weights):
    plt.text(i, v + 0.02, f"{v:.2f}", ha='center')
plt.show()

Optimizing Ensemble Weights (10000 trials)...
✓ Optimization Complete.
  Best Val Accuracy: 93.5632%
  Optimal Weights: [0.42092091 0.4683728  0.11070628]


In [36]:
# -----------------------------------------------------------
# CELL 8: FINAL COMPARISON (UPDATED)
# -----------------------------------------------------------
print("\n" + "="*60)
print("📊 FINAL PERFORMANCE COMPARISON")
print("="*60)

# 1. Prepare Test Predictions list
test_preds_list = [
    xgb_model.predict_proba(X_test),
    lgb_model.predict_proba(X_test),
    cat_model.predict_proba(X_test)
]

# 2. Get Ensemble Prediction
ensemble_pred = stacker.predict(test_preds_list)

# 3. Gather all predictions
preds = {
    "XGBoost": xgb_model.predict(X_test),
    "LightGBM": lgb_model.predict(X_test),
    "CatBoost": cat_model.predict(X_test),
    "Stacking Ensemble": ensemble_pred
}

# 4. Compare
accuracies = {}
print(f"{'Model':<20} | {'Accuracy':<10}")
print("-" * 35)

for name, p in preds.items():
    acc = get_accuracy(y_test, p)
    accuracies[name] = acc
    print(f"{name:<20} | {acc:.2%}")

# Winner Logic
winner = max(accuracies, key=accuracies.get)
print("-" * 35)
print(f"🏆 WINNER: {winner}")

# 5. Plot Confusion Matrix
print(f"\n>>> Detailed Report for {winner}:")
print_classification_report(y_test, preds[winner], class_names)

plot_confusion_matrix(
    get_confusion_matrix(y_test, preds[winner], NUM_CLASSES),
    class_names,
    f"{winner} Confusion Matrix"
)


📊 FINAL PERFORMANCE COMPARISON
Model                | Accuracy  
-----------------------------------
XGBoost              | 92.34%
LightGBM             | 92.34%
CatBoost             | 92.52%
Stacking Ensemble    | 93.25%
-----------------------------------
🏆 WINNER: Stacking Ensemble

>>> Detailed Report for Stacking Ensemble:
Class                     Precision  Recall     F1-Score  
------------------------------------------------------------
0                         0.7333     0.6471     0.6875
1                         0.9847     0.9877     0.9862
2                         0.7119     0.8077     0.7568
3                         0.9524     0.8333     0.8889
4                         0.8333     0.7895     0.8108
5                         0.9556     0.9451     0.9503
------------------------------------------------------------
Weighted Avg              0.9343     0.9325     0.9328


In [37]:
# -----------------------------------------------------------
# CELL 11: SAVE FINAL ENSEMBLE MODEL
# -----------------------------------------------------------
import json

print(f"Saving Ensemble to {MODEL_DIR}...")

# 1. Save the Stacker Object (The Optimizer)
stacker_path = os.path.join(MODEL_DIR, "final_weight_optimizer.pkl")
with open(stacker_path, "wb") as f:
    pickle.dump(stacker, f)

# 2. Save the Weights Configuration (Human Readable)
# We convert numpy types to python types for JSON compatibility
ensemble_config = {
    "model_weights": {
        "XGBoost": float(stacker.best_weights[0]),
        "LightGBM": float(stacker.best_weights[1]),
        "CatBoost": float(stacker.best_weights[2])
    },
    "best_validation_accuracy": float(stacker.best_acc),
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

config_path = os.path.join(MODEL_DIR, "ensemble_weights.json")
with open(config_path, "w") as f:
    json.dump(ensemble_config, f, indent=4)

print(f"✅ Optimizer Object saved: {stacker_path}")
print(f"✅ Weights Configuration saved: {config_path}")
print(f"   Weights: {ensemble_config['model_weights']}")

Saving Ensemble to /content/drive/MyDrive/FabricDefect/models...
✅ Optimizer Object saved: /content/drive/MyDrive/FabricDefect/models/final_weight_optimizer.pkl
✅ Weights Configuration saved: /content/drive/MyDrive/FabricDefect/models/ensemble_weights.json
   Weights: {'XGBoost': 0.4209209117330096, 'LightGBM': 0.4683728039015581, 'CatBoost': 0.11070628436543241}


In [40]:
# -----------------------------------------------------------
# CELL 12: COMPREHENSIVE BACKUP SYSTEM (FIXED)
# -----------------------------------------------------------
import shutil
import os
import pickle
import json
from datetime import datetime

print("\n" + "="*60)
print("💾 INITIATING FINAL BACKUP PROTOCOL")
print("="*60)

# 1. Create Timestamped Directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_folder = os.path.join(BACKUP_DIR, f"ensemble_run_{timestamp}")
os.makedirs(backup_folder, exist_ok=True)

print(f"📂 Backup Directory: {backup_folder}")

# 2. Define all models to backup
models_to_backup = {
    "CustomXGB": xgb_model,
    "CustomLGBM": lgb_model,
    "CustomCatB": cat_model,
    "WeightStacker": stacker
}

# 3. Save Objects
print("\nArchiving Model Objects...")
for name, model_obj in models_to_backup.items():
    filename = f"{name}.pkl"
    filepath = os.path.join(backup_folder, filename)

    try:
        with open(filepath, "wb") as f:
            pickle.dump(model_obj, f)

        size_kb = os.path.getsize(filepath) / 1024
        print(f"  ✅ Archived {name:15} ({size_kb:.2f} KB)")
    except Exception as e:
        print(f"  ❌ Failed to archive {name}: {e}")

# 4. Save Final Performance Report
print("\nSaving Final Report...")

# Retrieve the ensemble accuracy from the accuracies dict created in Cell 8
# If not found, recalculate it quickly
if 'final_acc' not in locals():
    if 'accuracies' in locals() and "Stacking Ensemble" in accuracies:
        final_acc = accuracies["Stacking Ensemble"]
    else:
        # Fallback calculation
        final_acc = get_accuracy(y_test, ensemble_pred)

report = {
    "winner": winner,
    "final_ensemble_accuracy": float(final_acc),
    "individual_accuracies": {k: float(v) for k, v in accuracies.items()},
    "optimal_weights": ensemble_config['model_weights']
}

report_path = os.path.join(backup_folder, "final_report.json")
with open(report_path, "w") as f:
    json.dump(report, f, indent=4)

print(f"  ✅ Report saved to final_report.json")

print("\n" + "="*60)
print("🎉 PROJECT BACKUP COMPLETE.")
print("="*60)


💾 INITIATING FINAL BACKUP PROTOCOL
📂 Backup Directory: /content/drive/MyDrive/FabricDefect/backups/ensemble_run_20251205_014432

Archiving Model Objects...
  ✅ Archived CustomXGB       (128.40 KB)
  ✅ Archived CustomLGBM      (128.63 KB)
  ✅ Archived CustomCatB      (129.95 KB)
  ✅ Archived WeightStacker   (0.28 KB)

Saving Final Report...
  ✅ Report saved to final_report.json

🎉 PROJECT BACKUP COMPLETE.
